In [ ]:
# ── CELLULE 1 : Chargement des données ──
import pandas as pd
import os, csv
import warnings
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Supprime le warning numpy longdouble (compatibilité plateforme)
warnings.filterwarnings("ignore", category=UserWarning, module="numpy")

Proj = "PRJNA755688-stage12"
path = "/mnt/g/ngs/data/" + Proj
path1 = path + "/"
path2 = path + "/rna/"
fichiers_non_trouve = []

f = path1 + "liste-files.csv"
df_tot = pd.DataFrame()

with open(f, 'r') as csvfile:
    filereader = csv.reader(csvfile, delimiter=',')
    for row in filereader:
        sample = row[0]
        label = row[1]

        f1 = path2 + "results-" + sample + ".txt"
        if os.path.isfile(f1):
            df = pd.read_csv(f1, sep="\t")
            df["sample"] = sample
            df_pivot = df.pivot_table(values=['count'], columns='RNA', index=['sample'])

            if "I" in label:
                label = "II"
            if "healthy" in label:
                label = "control"
            if "Normal" in label:
                label = "control"

            df_pivot["label"] = label
            df_tot = pd.concat([df_tot, df_pivot])
        else:
            fichiers_non_trouve.append(f1)

print("Fichiers non trouvés :", len(fichiers_non_trouve))
#print(df_tot.head())

encoder = LabelEncoder()
print(df_tot.label.unique())

print(df_tot.label.value_counts())

target = encoder.fit_transform(df_tot.label)
df_tot = df_tot.fillna(0)

Fichiers non trouvés : 0
['II' 'control']
label
control    561
II         399
Name: count, dtype: int64


In [3]:
# ── CELLULE 2 : Vérification labels ──
print(df_tot["label"])

sample
SRR17005923         II
SRR17005924         II
SRR17005930         II
SRR17005931         II
SRR17005945         II
                ...   
SRR16919803    control
SRR16919804    control
SRR16919805    control
SRR16919806    control
SRR16919807    control
Name: label, Length: 960, dtype: object


In [4]:
# ── CELLULE 3 : Vérification target ──
print(target)
print(target.shape)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 

In [31]:
# ── CELLULE 4 : Split train/test + équilibrage sur train UNIQUEMENT ──
from sklearn.model_selection import train_test_split

# Split stratifié AVANT équilibrage (évite data leakage)
labels = df_tot["label"].copy()
idx_train, idx_test = train_test_split(
    df_tot.index, test_size=0.20, random_state=222, stratify=labels
)

df_train_all = df_tot.loc[idx_train].copy()
y_train_all = labels.loc[idx_train]
df_test_all = df_tot.loc[idx_test].copy()
y_test_all = labels.loc[idx_test]

# Sauvegarder le test set pour évaluation future
X_test_final = df_test_all.reset_index(drop=True)
y_test_final = y_test_all.reset_index(drop=True)

# Équilibrer UNIQUEMENT le train (undersampling majoritaire)
class_counts = y_train_all.value_counts()
min_count = class_counts.min()

df_train_balanced = pd.DataFrame()
for cls in class_counts.index:
    df_cls = df_train_all[y_train_all == cls]
    if len(df_cls) > min_count:
        df_cls = df_cls.sample(n=min_count, random_state=42)
    df_train_balanced = pd.concat([df_train_balanced, df_cls])

df_tot = df_train_balanced.reset_index(drop=True)
target = encoder.fit_transform(df_tot["label"])

print("Distribution train après équilibrage :")
print(df_tot["label"].value_counts())
print(f"Train équilibré: {df_tot.shape}, Test réservé: {X_test_final.shape}")
print("[OK] Pas de data leakage : split avant équilibrage")

Distribution train après équilibrage :
label
control    319
II         319
Name: count, dtype: int64
Train équilibré: (638, 2653), Test réservé: (303, 2653)
[OK] Pas de data leakage : split avant équilibrage


In [32]:
# ── CELLULE 5 : Suppression colonne label ──
if "label" in df_tot.columns:
    df_tot = df_tot.drop(columns=["label"])
print(f"df_tot shape (sans label): {df_tot.shape}")

df_tot shape (sans label): (638, 2652)


In [33]:
# ── CELLULE 6 : Vérification shapes ──
print(f"df_tot shape (prêt pour ML): {df_tot.shape}")
print(f"target shape: {target.shape}")

df_tot shape (prêt pour ML): (638, 2652)
target shape: (638,)


In [ ]:
# ── CELLULE 7 : Diagnostic + filtrage des RNAs ──
import numpy as np

print("=" * 60)
print("DIAGNOSTIC AVANT FILTRAGE")
print("=" * 60)
print(f"Nb features : {df_tot.shape[1]}")
print(f"Nb échantillons:      {df_tot.shape[0]}")

zero_pct = (df_tot == 0).sum().sum() / (df_tot.shape[0] * df_tot.shape[1]) * 100
print(f"% de valeurs = 0:      {zero_pct:.1f}%")

nonzero_pct_per_rna = (df_tot > 0).sum(axis=0) / df_tot.shape[0]
rnas_exprimes = (nonzero_pct_per_rna > 0.20).sum()
print(f"RNAs exprimés dans >20%: {rnas_exprimes}/{df_tot.shape[1]}")

# Calcule la variance de chaque RNA (colonne) et trie par ordre décroissant
# 
variances = df_tot.var(axis=0).sort_values(ascending=False)
#print(f"\nTop 5 RNAs (variance max):")
#for rank, (idx, var) in enumerate(variances.head(5).items(), 1):
#    name = idx[1] if isinstance(idx, tuple) else idx  # idx[1] = nom du RNA (ex: hsa-miR-6131)
#    print(f"  {rank}. {name}: {var:.2e}")

print("\n" + "=" * 60)
print("FILTRAGE")
print("=" * 60)

# Filtre les colonnes RNA : garde celles exprimées dans ≥20% des échantillons
mask_exprime = nonzero_pct_per_rna >= 0.20
df_tot_f1 = df_tot.loc[:, mask_exprime]
print(f"Étape 1 (exprimés ≥20%): {df_tot.shape[1]} → {df_tot_f1.shape[1]}")

MAX_FEATURES = 50
if df_tot_f1.shape[1] > MAX_FEATURES:
    # Garde les 50 RNAs avec la plus forte variance (= les plus discriminants)
    variances_f1 = df_tot_f1.var(axis=0).sort_values(ascending=False)
    top_features = variances_f1.head(MAX_FEATURES).index  # index multi-niveaux : (count, nom_du_RNA)
    df_tot_filtered = df_tot_f1[top_features].copy()
    print(f"Étape 2 (top {MAX_FEATURES}): {df_tot_f1.shape[1]} → {df_tot_filtered.shape[1]}")
else:
    df_tot_filtered = df_tot_f1.copy()

print(f"Features retirées: {df_tot.shape[1] - df_tot_filtered.shape[1]}")
# Affiche les 10 RNAs les plus importants retenus pour le ML (top variance)
#print(f"Top 10 RNAs retenus:")
#for rank, idx in enumerate(variances_f1.head(10).index, 1):
#    name = idx[1] if isinstance(idx, tuple) else idx  # idx[1] = nom du RNA (ex: hsa-miR-6131)
#    print(f"  {rank}. {name}")
print("[OK] Prêt pour AutoGluon")

DIAGNOSTIC AVANT FILTRAGE
Nb features : 2652
Nb échantillons:      638
% de valeurs = 0:      69.6%
RNAs exprimés dans >20%: 2097/2652

FILTRAGE
Étape 1 (exprimés ≥20%): 2652 → 2097
Étape 2 (top 50): 2097 → 50
Features retirées: 2602
[OK] Prêt pour AutoGluon


In [35]:
# ── CELLULE 8 : AutoGluon avec PCA (medium_quality) ──
from autogluon.tabular import TabularPredictor
from sklearn.neighbors import NeighborhoodComponentsAnalysis
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import warnings
import os
warnings.filterwarnings("ignore")
# Supprime aussi le warning IProgress de tqdm
warnings.filterwarnings("ignore", category=UserWarning, module="tqdm")

try:
    X = df_tot_filtered.copy()
    print("[INFO] Données filtrées")
except NameError:
    X = df_tot.copy()
    print("[WARN] Données NON filtrées")

# Les colonnes sont au format MultiIndex : ( 'count', 'RNA' )
# Cette ligne aplatit les noms en une chaîne unique 
if isinstance(X.columns, pd.MultiIndex):
    X.columns = ['_'.join(str(c) for c in col) for col in X.columns]

y = encoder.inverse_transform(target)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=222)
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# NCA+PCA fit sur TRAIN uniquement
nca = NeighborhoodComponentsAnalysis(random_state=42)
X_train_nca = nca.fit_transform(X_train, encoder.transform(y_train))
X_test_nca = nca.transform(X_test)
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_nca)
X_test_pca = pca.transform(X_test_nca)
print(f"Composantes PCA: {pca.n_components_}")

df_train = pd.DataFrame(X_train_pca)
df_train.columns = df_train.columns.astype(str)
df_train["label"] = y_train
df_test = pd.DataFrame(X_test_pca)
df_test.columns = df_test.columns.astype(str)
df_test["label"] = y_test
print(f"Train PCA: {df_train.shape}, Test PCA: {df_test.shape}")

predictor = TabularPredictor(label="label", problem_type="multiclass", eval_metric="accuracy", verbosity=1).fit(
    train_data=df_train, time_limit=300, presets="medium_quality")

# Affiche le chemin relatif du modèle sauvegardé (par rapport au notebook)
path_rel = os.path.relpath(predictor.path)
print(f"[INFO] Modèle sauvegardé dans : {path_rel}")

print("\n" + "=" * 60)
print("LEADERBOARD (PCA)")
print("=" * 60)
leaderboard = predictor.leaderboard(df_test, silent=True)
print(leaderboard)

print("\n" + "=" * 60)
print("ÉVALUATION")
print("=" * 60)
perf = predictor.evaluate(df_test, auxiliary_metrics=True, silent=True)
for k, v in perf.items():
    print(f"  {k}: {v:.4f}")

print("\n" + "=" * 60)
print("FEATURE IMPORTANCE (composantes PCA)")
print("=" * 60)
importance = predictor.feature_importance(df_test)
print(importance)

No path specified. Models will be saved in: "AutogluonModels/ag-20260607_172743"


[INFO] Données filtrées
Train: (510, 50), Test: (128, 50)
Composantes PCA: 3
Train PCA: (510, 4), Test PCA: (128, 4)
[INFO] Modèle sauvegardé dans : AutogluonModels/ag-20260607_172743

LEADERBOARD (PCA)
                  model  score_test  score_val eval_metric  pred_time_test  \
0              LightGBM    1.000000        1.0    accuracy        0.001009   
1            LightGBMXT    1.000000        1.0    accuracy        0.001448   
2         LightGBMLarge    1.000000        1.0    accuracy        0.002054   
3              CatBoost    1.000000        1.0    accuracy        0.003901   
4        NeuralNetTorch    1.000000        1.0    accuracy        0.007771   
5               XGBoost    1.000000        1.0    accuracy        0.008988   
6      RandomForestEntr    1.000000        1.0    accuracy        0.072199   
7      RandomForestGini    1.000000        1.0    accuracy        0.074347   
8        ExtraTreesEntr    1.000000        1.0    accuracy        0.076008   
9        ExtraTre

In [36]:
# ── CELLULE 9 : AutoGluon sans PCA (medium_quality, vrai test set) ──
from autogluon.tabular import TabularPredictor
import warnings
warnings.filterwarnings("ignore")

try:
    df_ml = df_tot_filtered.copy()
    print("[INFO] Données filtrées")
except NameError:
    df_ml = df_tot.copy().drop(columns=["label"])

if isinstance(df_ml.columns, pd.MultiIndex):
    # Aplatit les noms MultiIndex
    df_ml.columns = ['_'.join(str(c) for c in col) for col in df_ml.columns]

df_ml["label"] = encoder.inverse_transform(target)
df_ml = df_ml.reset_index(drop=True)

try:
    X_test = X_test_final.copy()
    if isinstance(X_test.columns, pd.MultiIndex):
        X_test.columns = ['_'.join(str(c) for c in col) for col in X_test.columns]
    X_test["label"] = y_test_final.values
    X_test = X_test.reset_index(drop=True)
    test_data = X_test
    print(f"Train: {df_ml.shape}, Test réservé: {test_data.shape}")
except NameError:
    print("[WARN] X_test_final non défini")
    test_data = None

predictor_no_pca = TabularPredictor(label="label", problem_type="multiclass", eval_metric="accuracy", verbosity=1).fit(
    train_data=df_ml, time_limit=300, presets="medium_quality")

# Affiche le chemin relatif du modèle sauvegardé (par rapport au notebook)
import os
path_rel = os.path.relpath(predictor_no_pca.path)
print(f"[INFO] Modèle sauvegardé dans : {path_rel}")

print("\n" + "=" * 60)
print("LEADERBOARD (sans PCA)")
print("=" * 60)
if test_data is not None:
    leaderboard = predictor_no_pca.leaderboard(test_data, silent=True)
    print(leaderboard)

    print("\n" + "=" * 60)
    print("ÉVALUATION SUR VRAI TEST SET")
    print("=" * 60)
    perf = predictor_no_pca.evaluate(test_data, auxiliary_metrics=True, silent=True)
    for k, v in perf.items():
        print(f"  {k}: {v:.4f}")

    print("\n" + "=" * 60)
    #print("FEATURE IMPORTANCE (top 10)")
    #print("=" * 60)
    #importance = predictor_no_pca.feature_importance(test_data)
    #print(importance.head(10))
else:
    print("[SKIP] Test non disponible")

No path specified. Models will be saved in: "AutogluonModels/ag-20260607_172823"


[INFO] Données filtrées
Train: (638, 51), Test réservé: (303, 2654)
[INFO] Modèle sauvegardé dans : AutogluonModels/ag-20260607_172823

LEADERBOARD (sans PCA)
                  model  score_test  score_val eval_metric  pred_time_test  \
0         LightGBMLarge    1.000000   1.000000    accuracy        0.001884   
1              LightGBM    1.000000   1.000000    accuracy        0.002130   
2              CatBoost    1.000000   1.000000    accuracy        0.008621   
3        NeuralNetTorch    1.000000   1.000000    accuracy        0.030946   
4      RandomForestEntr    1.000000   1.000000    accuracy        0.084993   
5      RandomForestGini    1.000000   1.000000    accuracy        0.092138   
6        ExtraTreesEntr    1.000000   1.000000    accuracy        0.093954   
7        ExtraTreesGini    1.000000   1.000000    accuracy        0.101259   
8               XGBoost    0.996700   1.000000    accuracy        0.009450   
9   WeightedEnsemble_L2    0.996700   1.000000    accuracy   

In [27]:
# ── CELLULE 10 : AutoGluon medium_quality etendu (sans Ray, sans bruit) ──
import os
os.environ["PYTHONWARNINGS"] = "ignore"

from autogluon.tabular import TabularPredictor
import warnings
warnings.filterwarnings("ignore")

try:
    df_ml = df_tot_filtered.copy()
except NameError:
    df_ml = df_tot.copy().drop(columns=["label"])

if isinstance(df_ml.columns, pd.MultiIndex):
    df_ml.columns = ['_'.join(str(c) for c in col) for col in df_ml.columns]

df_ml["label"] = encoder.inverse_transform(target)
df_ml = df_ml.reset_index(drop=True)

try:
    X_test = X_test_final.copy()
    if isinstance(X_test.columns, pd.MultiIndex):
        X_test.columns = ['_'.join(str(c) for c in col) for col in X_test.columns]
    X_test["label"] = y_test_final.values
    X_test = X_test.reset_index(drop=True)
    test_data = X_test
    print(f"Train: {df_ml.shape}, Test: {test_data.shape}")
except NameError:
    test_data = None

predictor_best = TabularPredictor(
    label="label", problem_type="multiclass", eval_metric="accuracy", verbosity=1
).fit(train_data=df_ml, time_limit=600, presets="medium_quality")

model_name = os.path.basename(predictor_best.path)
print(f"[INFO] Modele: {model_name}")

print("\n" + "=" * 60)
print("LEADERBOARD")
print("=" * 60)
if test_data is not None:
    leaderboard = predictor_best.leaderboard(test_data, silent=True)
    print(leaderboard)

    print("\n" + "=" * 60)
    print("EVALUATION")
    print("=" * 60)
    perf = predictor_best.evaluate(test_data, auxiliary_metrics=True, silent=True)
    for k, v in perf.items():
        print(f"  {k}: {v:.4f}")

    print("\n" + "=" * 60)
    print("MATRICE DE CONFUSION")
    print("=" * 60)
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(test_data["label"], predictor_best.predict(test_data))
    print(cm)
else:
    print("[SKIP] Test non disponible")

No path specified. Models will be saved in: "AutogluonModels/ag-20260607_144544"


Train: (510, 51), Test: (128, 2654)
[INFO] Modele: ag-20260607_144544

LEADERBOARD
                  model  score_test  score_val eval_metric  pred_time_test  \
0              LightGBM    1.000000        1.0    accuracy        0.001416   
1         LightGBMLarge    1.000000        1.0    accuracy        0.001662   
2              CatBoost    1.000000        1.0    accuracy        0.005393   
3               XGBoost    1.000000        1.0    accuracy        0.008788   
4        NeuralNetTorch    1.000000        1.0    accuracy        0.027818   
5        ExtraTreesEntr    1.000000        1.0    accuracy        0.084431   
6      RandomForestEntr    1.000000        1.0    accuracy        0.085625   
7        ExtraTreesGini    1.000000        1.0    accuracy        0.086158   
8      RandomForestGini    1.000000        1.0    accuracy        0.090961   
9            LightGBMXT    0.992188        1.0    accuracy        0.002128   
10      NeuralNetFastAI    0.992188        1.0    accuracy 